In [1]:

import pandas as pd
import numpy as np
import plotly.express as px

from pathlib import Path
from tqdm.auto import tqdm

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)


PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data" / "raw"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed"

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)


file_path = DATA_PATH / "DataCoSupplyChainDataset.csv"

df = pd.read_csv(
    file_path,
    encoding="latin1",
    low_memory=False
)

print("Dataset successfully loaded!")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

# Dataset dimensions
print("Dataset Shape:", df.shape)

# First five records
display(df.head())

# Random five records
display(df.sample(5, random_state=42))

# Column names
print("\nColumn Names:")
for column in df.columns:
    print(column)

# Data types and non-null counts

df.info()

# Numerical variables
numerical_columns = df.select_dtypes(
    include=np.number
).columns.tolist()

# Categorical variables
categorical_columns = df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("\nNumerical Columns:", len(numerical_columns))
print("Categorical Columns:", len(categorical_columns))

missing_analysis = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (
        df.isnull().mean() * 100
    )
})

missing_analysis = missing_analysis.sort_values(
    by="Missing_Percentage",
    ascending=False
)

display(missing_analysis)

missing_data = missing_analysis[
    missing_analysis["Missing_Count"] > 0
].reset_index(names="Column")

fig = px.bar(
    missing_data,
    x="Column",
    y="Missing_Percentage",
    title="Missing Values by Column (%)",
    labels={
        "Missing_Percentage": "Missing Values (%)",
        "Column": "Dataset Columns"
    }
)

fig.update_layout(
    xaxis_tickangle=-45,
    template="plotly_white"
)

fig.show()

duplicate_count = df.duplicated().sum()

print("Total Duplicate Records:", duplicate_count)

print(
    "Duplicate Percentage:",
    round((duplicate_count / len(df)) * 100, 2),
    "%"
)

numerical_summary = df.describe().T

numerical_summary["Missing_Count"] = (
    df[numerical_columns].isnull().sum()
)

display(numerical_summary)


date_columns = [
    col for col in df.columns
    if "date" in col.lower()
]

for col in date_columns:
    print(f"\n{col}")
    print("Data type:", df[col].dtype)
    print("Sample values:")
    print(df[col].head(5).tolist())



df_quality = df.copy()

# Check key identifier uniqueness
identifier_columns = [
    "Order Item Id",
    "Order Id",
    "Customer Id",
    "Product Card Id"
]

identifier_results = []

for col in identifier_columns:
    identifier_results.append({
        "Column": col,
        "Unique_Values": df_quality[col].nunique(),
        "Duplicate_Count": df_quality[col].duplicated().sum(),
        "Missing_Count": df_quality[col].isnull().sum()
    })

identifier_audit = pd.DataFrame(identifier_results)

display(identifier_audit)

# ==========================================
# MISSING VALUE ASSESSMENT
# ==========================================

missing_audit = pd.DataFrame({
    "Missing_Count": df_quality.isnull().sum(),
    "Missing_Percentage": df_quality.isnull().mean() * 100
})

missing_audit = missing_audit.sort_values(
    "Missing_Percentage",
    ascending=False
)

display(missing_audit[missing_audit["Missing_Count"] > 0])

# Identify columns with 100% missing values
fully_missing_columns = missing_audit[
    missing_audit["Missing_Percentage"] == 100
].index.tolist()

print("Fully missing columns:", fully_missing_columns)

# Remove only completely empty columns from working copy
df_quality.drop(
    columns=fully_missing_columns,
    inplace=True
)

# ==========================================
# NUMERICAL INTEGRITY
# ==========================================

numeric_cols = [
    "Sales",
    "Order Item Total",
    "Order Item Product Price",
    "Product Price",
    "Order Item Quantity",
    "Order Item Discount",
    "Order Item Discount Rate",
    "Order Profit Per Order",
    "Benefit per order",
    "Order Item Profit Ratio",
    "Days for shipping (real)",
    "Days for shipment (scheduled)"
]

numeric_audit = []

for col in numeric_cols:

    if col not in df_quality.columns:
        continue

    values = pd.to_numeric(
        df_quality[col],
        errors="coerce"
    )

    numeric_audit.append({
        "Column": col,
        "Minimum": values.min(),
        "Maximum": values.max(),
        "Mean": values.mean(),
        "Zero_Count": (values == 0).sum(),
        "Negative_Count": (values < 0).sum(),
        "Invalid_Count": values.isna().sum()
    })

numeric_audit = pd.DataFrame(numeric_audit)

display(numeric_audit)

# ==========================================
# DATE INTEGRITY
# ==========================================

order_date_col = "order date (DateOrders)"
shipping_date_col = "shipping date (DateOrders)"

df_quality[order_date_col] = pd.to_datetime(
    df_quality[order_date_col],
    errors="coerce"
)

df_quality[shipping_date_col] = pd.to_datetime(
    df_quality[shipping_date_col],
    errors="coerce"
)

# Calculate actual difference between dates
df_quality["Calculated Shipping Days"] = (
    df_quality[shipping_date_col] -
    df_quality[order_date_col]
).dt.total_seconds() / 86400

# Compare with reported shipping duration
df_quality["Shipping Days Difference"] = (
    df_quality["Calculated Shipping Days"] -
    df_quality["Days for shipping (real)"]
)

date_audit = {
    "Invalid Order Dates": df_quality[order_date_col].isna().sum(),
    "Invalid Shipping Dates": df_quality[shipping_date_col].isna().sum(),

    "Shipping Before Order": (
        df_quality["Calculated Shipping Days"] < 0
    ).sum(),

    "Order Date Range": (
        df_quality[order_date_col].min(),
        df_quality[order_date_col].max()
    ),

    "Shipping Date Range": (
        df_quality[shipping_date_col].min(),
        df_quality[shipping_date_col].max()
    ),

    "Shipping Duration Mismatch": (
        df_quality["Shipping Days Difference"].abs() > 0.01
    ).sum()
}

for key, value in date_audit.items():
    print(f"{key}: {value}")

# ==========================================
# BUSINESS CONSISTENCY CHECKS
# ==========================================

# Calculate gross line value
df_quality["Calculated Gross Value"] = (
    df_quality["Order Item Quantity"] *
    df_quality["Order Item Product Price"]
)

# Calculate expected discounted line value
df_quality["Calculated Net Value"] = (
    df_quality["Calculated Gross Value"] -
    df_quality["Order Item Discount"]
)

# Compare calculated values against reported values
df_quality["Net Value Difference"] = (
    df_quality["Calculated Net Value"] -
    df_quality["Order Item Total"]
)

# Check late delivery flag against actual and scheduled duration
df_quality["Calculated Late Risk"] = (
    df_quality["Days for shipping (real)"] >
    df_quality["Days for shipment (scheduled)"]
).astype(int)

business_checks = pd.DataFrame({
    "Check": [
        "Gross value calculation",
        "Net order item value calculation",
        "Late delivery risk consistency"
    ],

    "Mismatch_Count": [
        (
            df_quality["Calculated Gross Value"] -
            df_quality["Sales"]
        ).abs().gt(0.01).sum(),

        df_quality["Net Value Difference"].abs().gt(0.01).sum(),

        (
            df_quality["Calculated Late Risk"] !=
            df_quality["Late_delivery_risk"]
        ).sum()
    ]
})

display(business_checks)

# ==========================================
# CATEGORICAL CONSISTENCY
# ==========================================

categorical_checks = [
    "Order Status",
    "Delivery Status",
    "Shipping Mode",
    "Market",
    "Order Region",
    "Customer Segment",
    "Department Name",
    "Category Name",
    "Type"
]

for col in categorical_checks:

    print(f"\n{'=' * 50}")
    print(f"Column: {col}")
    print(f"Unique values: {df_quality[col].nunique()}")

    display(
        df_quality[col]
        .value_counts(dropna=False)
        .head(20)
        .to_frame("Record_Count")
    )

# ==========================================
# EXPORT QUALITY ASSESSMENT
# ==========================================

audit_path = OUTPUT_PATH / "data_quality_assessment.xlsx"

with pd.ExcelWriter(audit_path, engine="openpyxl") as writer:

    missing_audit.to_excel(
        writer,
        sheet_name="Missing_Values"
    )

    identifier_audit.to_excel(
        writer,
        sheet_name="Identifiers",
        index=False
    )

    numeric_audit.to_excel(
        writer,
        sheet_name="Numerical_Audit",
        index=False
    )

    business_checks.to_excel(
        writer,
        sheet_name="Business_Checks",
        index=False
    )

print("Quality assessment exported successfully!")
print(audit_path)

# 1. Investigate net order item value mismatches

net_mismatch = df_quality[
    df_quality["Net Value Difference"].abs() > 0.01
]

print("Net Value Mismatches:", len(net_mismatch))

display(
    net_mismatch[
        [
            "Order Id",
            "Order Item Id",
            "Order Item Quantity",
            "Order Item Product Price",
            "Order Item Discount",
            "Order Item Total",
            "Calculated Net Value",
            "Net Value Difference"
        ]
    ].head(15)
)


# 2. Investigate late delivery risk mismatches

risk_mismatch = df_quality[
    df_quality["Calculated Late Risk"] !=
    df_quality["Late_delivery_risk"]
]

print("Late Delivery Risk Mismatches:", len(risk_mismatch))

display(
    risk_mismatch[
        [
            "Order Id",
            "Days for shipping (real)",
            "Days for shipment (scheduled)",
            "Late_delivery_risk",
            "Calculated Late Risk",
            "Delivery Status"
        ]
    ].head(15)
)

# Compare the recorded and calculated risk values
display(
    pd.crosstab(
        df_quality["Late_delivery_risk"],
        df_quality["Calculated Late Risk"],
        rownames=["Recorded Risk"],
        colnames=["Calculated Risk"]
    )
)

date_audit = {
    "Invalid Order Dates": df_quality[
        "order date (DateOrders)"
    ].isna().sum(),

    "Invalid Shipping Dates": df_quality[
        "shipping date (DateOrders)"
    ].isna().sum(),

    "Shipping Before Order": (
        df_quality["Calculated Shipping Days"] < 0
    ).sum(),

    "Shipping Duration Mismatches": (
        df_quality["Shipping Days Difference"].abs() > 0.01
    ).sum(),

    "Earliest Order": df_quality[
        "order date (DateOrders)"
    ].min(),

    "Latest Order": df_quality[
        "order date (DateOrders)"
    ].max()
}

for key, value in date_audit.items():
    print(f"{key}: {value}")

# Check delivery status among risk mismatches

display(
    risk_mismatch["Delivery Status"]
    .value_counts(dropna=False)
)

# Compare recorded and calculated risk by delivery status

display(
    pd.crosstab(
        df_quality["Delivery Status"],
        [
            df_quality["Late_delivery_risk"],
            df_quality["Calculated Late Risk"]
        ]
    )
)

# Round differences to two decimal places before counting

df_quality["Rounded Net Value Difference"] = (
    df_quality["Net Value Difference"].round(2)
)

print(
    "Differences greater than ₹0.01:",
    (df_quality["Rounded Net Value Difference"].abs() > 0.01).sum()
)

print(
    "Differences equal to ₹0.01:",
    (df_quality["Rounded Net Value Difference"].abs() == 0.01).sum()
)

print("Invalid order dates:",
      df_quality["order date (DateOrders)"].isna().sum())

print("Invalid shipping dates:",
      df_quality["shipping date (DateOrders)"].isna().sum())

print("Shipping before order:",
      (df_quality["Calculated Shipping Days"] < 0).sum())

print("Shipping duration mismatches:",
      (df_quality["Shipping Days Difference"].abs() > 0.01).sum())

print("Order date range:",
      df_quality["order date (DateOrders)"].min(),
      "to",
      df_quality["order date (DateOrders)"].max())

# Calculate elapsed duration in days
df_quality["Elapsed Shipping Days"] = (
    df_quality["shipping date (DateOrders)"]
    - df_quality["order date (DateOrders)"]
).dt.total_seconds() / 86400

# Calculate difference between calendar dates only
df_quality["Calendar Shipping Days"] = (
    df_quality["shipping date (DateOrders)"].dt.normalize()
    - df_quality["order date (DateOrders)"].dt.normalize()
).dt.days

# Compare both calculations with the recorded duration
df_quality["Elapsed Duration Difference"] = (
    df_quality["Elapsed Shipping Days"]
    - df_quality["Days for shipping (real)"]
)

df_quality["Calendar Duration Difference"] = (
    df_quality["Calendar Shipping Days"]
    - df_quality["Days for shipping (real)"]
)

print("Elapsed duration difference distribution:")
display(
    df_quality["Elapsed Duration Difference"]
    .round(2).value_counts().sort_index()
)

print("Calendar duration mismatch count:",
      (df_quality["Calendar Duration Difference"] != 0).sum())

print("Elapsed duration mismatch count:",
      (df_quality["Elapsed Duration Difference"].abs() > 0.01).sum())

date_mismatch = df_quality[
    df_quality["Shipping Days Difference"].abs() > 0.01
]

display(
    date_mismatch[
        [
            "Order Id",
            "order date (DateOrders)",
            "shipping date (DateOrders)",
            "Days for shipping (real)",
            "Elapsed Shipping Days",
            "Calendar Shipping Days",
            "Delivery Status"
        ]
    ].head(20)
)

# Create a separate working copy
df_clean = df.copy()

# Remove fully missing columns
fully_missing = df_clean.columns[
    df_clean.isna().all()
].tolist()

df_clean.drop(columns=fully_missing, inplace=True)

# Remove unnecessary and personally identifiable columns
columns_to_remove = [
    "Order Zipcode",
    "Customer Lname",
    "Customer Zipcode",
    "Customer Email",
    "Customer Password",
    "Customer Fname",
    "Customer Street"
]

df_clean.drop(
    columns=[
        col for col in columns_to_remove
        if col in df_clean.columns
    ],
    inplace=True
)

# Convert date columns
df_clean["order date (DateOrders)"] = pd.to_datetime(
    df_clean["order date (DateOrders)"],
    errors="coerce"
)

df_clean["shipping date (DateOrders)"] = pd.to_datetime(
    df_clean["shipping date (DateOrders)"],
    errors="coerce"
)

# Create calendar shipping duration
df_clean["Calendar Shipping Days"] = (
    df_clean["shipping date (DateOrders)"].dt.normalize()
    - df_clean["order date (DateOrders)"].dt.normalize()
).dt.days

# Create a separate calculated late-delivery indicator
df_clean["Calculated Late Delivery"] = (
    df_clean["Days for shipping (real)"]
    > df_clean["Days for shipment (scheduled)"]
).astype("int8")

# Create order year and month
df_clean["Order Year"] = (
    df_clean["order date (DateOrders)"].dt.year
)

df_clean["Order Month"] = (
    df_clean["order date (DateOrders)"].dt.month
)

df_clean["Order Year Month"] = (
    df_clean["order date (DateOrders)"]
    .dt.strftime("%Y-%m")
)

# Check final dataset
print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)
print("Missing values:", df_clean.isna().sum().sum())
print("Duplicate rows:", df_clean.duplicated().sum())

display(df_clean.head())

clean_path = OUTPUT_PATH / "supply_chain_clean.csv"

df_clean.to_csv(
    clean_path,
    index=False,
    encoding="utf-8-sig"
)

print("Cleaned dataset saved successfully!")
print("Location:", clean_path)
print("Rows:", len(df_clean))
print("Columns:", len(df_clean.columns))
print(
    "File size (MB):",
    round(clean_path.stat().st_size / (1024 * 1024), 2)
)

df_test = pd.read_csv(clean_path)

print("Original cleaned shape:", df_clean.shape)
print("Reloaded CSV shape:", df_test.shape)

if df_clean.shape == df_test.shape:
    print("SUCCESS: Dataset saved and verified.")
else:
    print("WARNING: Shape mismatch. Investigate before proceeding.")

C:\Users\roder\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset successfully loaded!
Rows: 180,519
Columns: 53
Dataset Shape: (180519, 53)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.25,314.64,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Cally,20755,Holloway,XXXXXXXXX,Consumer,PR,5365 Noble Nectar Island,725.00,2,Fitness,18.25,-66.04,Pacific Asia,Bekasi,Indonesia,20755,1/31/2018 22:56,77202,1360,13.11,0.04,180517,327.75,0.29,1,327.75,314.64,91.25,Southeast Asia,Java Occidental,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.09,311.36,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Irene,19492,Luna,XXXXXXXXX,Consumer,PR,2679 Rustic Loop,725.00,2,Fitness,18.28,-66.04,Pacific Asia,Bikaner,India,19492,1/13/2018 12:27,75939,1360,16.39,0.05,179254,327.75,-0.80,1,327.75,311.36,-249.09,South Asia,Rajastán,PENDING,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.78,309.72,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,XXXXXXXXX,Gillian,19491,Maldonado,XXXXXXXXX,Consumer,CA,8510 Round Bear Gate,"95,125.00",2,Fitness,37.29,-121.88,Pacific Asia,Bikaner,India,19491,1/13/2018 12:06,75938,1360,18.03,0.06,179253,327.75,-0.80,1,327.75,309.72,-247.78,South Asia,Rajastán,CLOSED,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.86,304.81,Advance shipping,0,73,Sporting Goods,Los Angeles,EE. UU.,XXXXXXXXX,Tana,19490,Tate,XXXXXXXXX,Home Office,CA,3200 Amber Bend,"90,027.00",2,Fitness,34.13,-118.29,Pacific Asia,Townsville,Australia,19490,1/13/2018 11:45,75937,1360,22.94,0.07,179252,327.75,0.08,1,327.75,304.81,22.86,Oceania,Queensland,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.21,298.25,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Orli,19489,Hendricks,XXXXXXXXX,Corporate,PR,8671 Iron Anchor Corners,725.00,2,Fitness,18.25,-66.04,Pacific Asia,Townsville,Australia,19489,1/13/2018 11:24,75936,1360,29.50,0.09,179251,327.75,0.45,1,327.75,298.25,134.21,Oceania,Queensland,PENDING_PAYMENT,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
80120,TRANSFER,5,4,11.09,175.99,Late delivery,1,48,Water Sports,Caguas,Puerto Rico,XXXXXXXXX,Zachary,6109,Tate,XXXXXXXXX,Consumer,PR,8916 Round Zephyr Ridge,725.00,7,Fan Shop,18.20,-66.37,USCA,Los Angeles,Estados Unidos,6109,4/1/2016 21:05,31299,1073,24.00,0.12,78239,199.99,0.06,1,199.99,175.99,11.09,West of USA,California,PROCESSING,"90,036.00",1073,48,NaN,http://images.acmesports.sports/Pelican+Sunstr...,Pelican Sunstream 100 Kayak,199.99,0,4/6/2016 21:05,Standard Class
19670,PAYMENT,2,1,9.80,245.00,Late delivery,1,24,Women's Apparel,Florissant,EE. UU.,XXXXXXXXX,Gary,12041,Oneal,XXXXXXXXX,Consumer,MO,372 Sunny Arbor,"63,033.00",5,Golf,38.81,-90.30,LATAM,Tegucigalpa,Honduras,12041,6/9/2017 18:43,61023,502,5.00,0.02,152658,50.00,0.04,5,250.00,245.00,9.80,Central America,Francisco Morazán,PENDING_PAYMENT,NaN,502,24,NaN,http://images.acmesports.sports/Nike+Men%27s+D...,Nike Men's Dri-FIT Victory Golf Polo,50.00,0,6/11/2017 18:43,First Class
114887,TRANSFER,2,4,117.55,244.90,Advance shipping,0,46,Indoor/Outdoor Games,Carmichael,EE. UU.,XXXXXXXXX,Andrea,4271,Alexander,XXXXXXXXX,Corporate,CA,2407 Foggy Acres,"95,608.00",7,Fan Shop,38.61,-121.32,LATAM,Tijuana,México,4271,1/2/2015 14:32,111,1014,5.00,0.02,253,49.98,0.48,5,249.90,244.90,117.55,Central America,Baja California,PROCESSING,NaN,1014,46,NaN,http://images.acmesports.sports/O%27Brien+Men%...,O'Brien Men's Neoprene Life Vest,49.98,0,1/4/2015 14:32,Standard Class
120110,TRANSFER,5,4,118.43,251.98,Late delivery,1,43,Camping & Hiking,Troy,EE. UU.,XXXXXXXXX,Joyce,3521,Callahan,XXXXXXXXX,Consumer,NY,4075 Silver Hills Heights,"12,180.00",7,Fan Shop,42.73,-73.69,Africa,Pretoria,SudAfrica,3521,1/10/2017 1:46,50699,957,48.00,0.16,126700,299.98,0.47,1,299.98,251.98,118.43,Southern Africa,Gauteng,PENDING,NaN,957,43,NaN,http://images.acmesports.sports/Diamondback+Wo...,Diamondback Women's Serene Classic Comfort Bi,299.98,0,1/15/2017 1:46,Standard Class
56658,DEBIT,2,4,-21.59,107.97,Advance shipping,0,29,Shop By Sport,Caguas,Puerto Rico,XXXXXXXXX,Mildred,12073,Merritt,XXXXXXXXX,Consumer,PR,4661 Green Freeway,725.00,5,Golf,18.23,-66.37,LATAM,Santiago de los Caballeros,República Dominicana,12073,4/6/2017 7:15,56606,627,12.00,0.10,141586,39.99,-0.20,3,119.97,107.97,-21.59,Caribbean,Santiago de Chile,COMPLETE,NaN,627,29,NaN,http://images.acmesports.sports/Under+Armour+G...,Under Armour Girls' Toddler Spine Surge Runni,39.99,0,4/8/2017 7:15,Standard Class



Column Names:
Type
Days for shipping (real)
Days for shipment (scheduled)
Benefit per order
Sales per customer
Delivery Status
Late_delivery_risk
Category Id
Category Name
Customer City
Customer Country
Customer Email
Customer Fname
Customer Id
Customer Lname
Customer Password
Customer Segment
Customer State
Customer Street
Customer Zipcode
Department Id
Department Name
Latitude
Longitude
Market
Order City
Order Country
Order Customer Id
order date (DateOrders)
Order Id
Order Item Cardprod Id
Order Item Discount
Order Item Discount Rate
Order Item Id
Order Item Product Price
Order Item Profit Ratio
Order Item Quantity
Sales
Order Item Total
Order Profit Per Order
Order Region
Order State
Order Status
Order Zipcode
Product Card Id
Product Category Id
Product Description
Product Image
Product Name
Product Price
Product Status
shipping date (DateOrders)
Shipping Mode
<class 'pandas.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column           

C:\Users\roder\AppData\Local\Temp\ipykernel_1988\94288843.py:57: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(


... (39 outputs left out)

,Order Id,order date (DateOrders),shipping date (DateOrders),Days for shipping (real),Elapsed Shipping Days,Calendar Shipping Days,Delivery Status
19,75921,2018-01-13 06:09:00,2018-01-13 18:09:00,0,0.50,0,Shipping on time
20,75920,2018-01-13 05:48:00,2018-01-13 17:48:00,0,0.50,0,Shipping on time
38,75902,2018-01-12 23:30:00,2018-01-13 11:30:00,1,0.50,1,Late delivery
39,75901,2018-01-12 23:09:00,2018-01-13 11:09:00,1,0.50,1,Shipping canceled
40,75900,2018-01-12 22:48:00,2018-01-13 10:48:00,1,0.50,1,Late delivery
41,75899,2018-01-12 22:27:00,2018-01-13 10:27:00,1,0.50,1,Late delivery
387,75867,2018-01-12 11:14:00,2018-01-12 23:14:00,0,0.50,0,Shipping on time
388,75866,2018-01-12 10:53:00,2018-01-12 22:53:00,0,0.50,0,Shipping on time
389,75865,2018-01-12 10:32:00,2018-01-12 22:32:00,0,0.50,0,Shipping on time
447,75807,2018-01-11 14:13:00,2018-01-12 02:13:00,1,0.50,1,Late delivery


Original shape: (180519, 53)
Cleaned shape: (180519, 50)
Missing values: 0
Duplicate rows: 0


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Id,Customer Segment,Customer State,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Product Card Id,Product Category Id,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode,Calendar Shipping Days,Calculated Late Delivery,Order Year,Order Month,Order Year Month
0,DEBIT,3,4,91.25,314.64,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,20755,Consumer,PR,2,Fitness,18.25,-66.04,Pacific Asia,Bekasi,Indonesia,20755,2018-01-31 22:56:00,77202,1360,13.11,0.04,180517,327.75,0.29,1,327.75,314.64,91.25,Southeast Asia,Java Occidental,COMPLETE,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-02-03 22:56:00,Standard Class,3,0,2018,1,2018-01
1,TRANSFER,5,4,-249.09,311.36,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,19492,Consumer,PR,2,Fitness,18.28,-66.04,Pacific Asia,Bikaner,India,19492,2018-01-13 12:27:00,75939,1360,16.39,0.05,179254,327.75,-0.80,1,327.75,311.36,-249.09,South Asia,Rajastán,PENDING,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-18 12:27:00,Standard Class,5,1,2018,1,2018-01
2,CASH,4,4,-247.78,309.72,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,19491,Consumer,CA,2,Fitness,37.29,-121.88,Pacific Asia,Bikaner,India,19491,2018-01-13 12:06:00,75938,1360,18.03,0.06,179253,327.75,-0.80,1,327.75,309.72,-247.78,South Asia,Rajastán,CLOSED,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-17 12:06:00,Standard Class,4,0,2018,1,2018-01
3,DEBIT,3,4,22.86,304.81,Advance shipping,0,73,Sporting Goods,Los Angeles,EE. UU.,19490,Home Office,CA,2,Fitness,34.13,-118.29,Pacific Asia,Townsville,Australia,19490,2018-01-13 11:45:00,75937,1360,22.94,0.07,179252,327.75,0.08,1,327.75,304.81,22.86,Oceania,Queensland,COMPLETE,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-16 11:45:00,Standard Class,3,0,2018,1,2018-01
4,PAYMENT,2,4,134.21,298.25,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,19489,Corporate,PR,2,Fitness,18.25,-66.04,Pacific Asia,Townsville,Australia,19489,2018-01-13 11:24:00,75936,1360,29.50,0.09,179251,327.75,0.45,1,327.75,298.25,134.21,Oceania,Queensland,PENDING_PAYMENT,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-15 11:24:00,Standard Class,2,0,2018,1,2018-01


Cleaned dataset saved successfully!
Location: c:\Users\roder\Downloads\Project\Supply-Chain-Consulting\data\processed\supply_chain_clean.csv
Rows: 180519
Columns: 50
File size (MB): 86.2
Original cleaned shape: (180519, 50)
Reloaded CSV shape: (180519, 50)
SUCCESS: Dataset saved and verified.
